# Size Evolution Analysis Notebook

This notebook analyzes the size evolution results from the consolidated analysis.
The size evolution model is: **log₁₀(R_e/kpc) = A - α log(1 + z)** at fixed stellar mass.

## Results from consolidated_analysis.py

The consolidated analysis produces size evolution fits for different galaxy samples:
- COWLS (full sample)
- COWLS Early-type (Sersic ≤ 2.5)
- COWLS Late-type (Sersic > 2.5)
- CWMGs (COSMOS WEB Massive Galaxies, full sample)
- CWMGs Early-type
- CWMGs Late-type


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set up plotting
plt.style.use('default')
sns.set_palette("husl")

# Set figure parameters
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = False


In [ ]:
# Load the size evolution results from consolidated analysis
results_file = 'size_evolution_from_binned_fits.csv'

if Path(results_file).exists():
    df_results = pd.read_csv(results_file)
    print("Size Evolution Results from Consolidated Analysis:")
    print("=" * 60)
    print(df_results.round(3))
    print("\nModel: log₁₀(R_e/kpc) = A - α log(1 + z) at fixed stellar mass")
else:
    print(f"Results file {results_file} not found.")
    df_results = None


## Key Results Summary

The size evolution analysis reveals interesting patterns:

### Physical Interpretation:
- **A (normalization)**: log₁₀ effective radius at z=0
- **α (evolution parameter)**: How size changes with redshift
  - **α > 0**: Galaxies were smaller in the past (size growth)
  - **α < 0**: Galaxies were larger in the past (size shrinking)
  - **α ≈ 0**: No significant size evolution

### Key Findings:
1. **COWLS lenses** show weak size evolution (α ≈ -0.2)
2. **CWMGs** show very weak size evolution (α ≈ -0.08)
3. **Early-type galaxies** show minimal size evolution
4. **Late-type galaxies** show more significant size evolution (α ≈ -0.3 for CWMGs Late-type)

### Sample Characteristics:
- **COWLS**: Higher mass galaxies (M* ≈ 10.2) with weak evolution
- **CWMGs**: Lower mass galaxies (M* ≈ 10.0) with minimal evolution
- **Early-type**: More compact galaxies with stable sizes
- **Late-type**: More extended galaxies with some size evolution


In [ ]:
# Create comprehensive analysis plots
if df_results is not None:
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # Plot 1: Size evolution parameters comparison
    samples = df_results['sample']
    A_values = df_results['A']
    A_errors = df_results['A_err']
    alpha_values = df_results['alpha']
    alpha_errors = df_results['alpha_err']
    
    # Color code by sample type
    colors = []
    for sample in samples:
        if 'COWLS' in sample:
            colors.append('red' if 'Early' in sample else 'blue' if 'Late' in sample else 'darkred')
        else:  # CWMGs
            colors.append('orange' if 'Early' in sample else 'green' if 'Late' in sample else 'darkorange')
    
    # Plot A parameter
    bars1 = ax1.bar(range(len(samples)), A_values, yerr=A_errors, capsize=5, 
                    color=colors, alpha=0.7, edgecolor='black')
    ax1.set_xlabel('Sample')
    ax1.set_ylabel('A (log₁₀ R_e/kpc at z=0)')
    ax1.set_title('Size Evolution Normalization')
    ax1.set_xticks(range(len(samples)))
    ax1.set_xticklabels(samples, rotation=45, ha='right')
    ax1.grid(False)
    
    # Add value labels on bars
    for i, (bar, val) in enumerate(zip(bars1, A_values)):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                f'{val:.2f}', ha='center', va='bottom', fontsize=9)
    
    # Plot α parameter
    bars2 = ax2.bar(range(len(samples)), alpha_values, yerr=alpha_errors, capsize=5,
                    color=colors, alpha=0.7, edgecolor='black')
    ax2.set_xlabel('Sample')
    ax2.set_ylabel('α (size evolution parameter)')
    ax2.set_title('Size Evolution Parameter')
    ax2.set_xticks(range(len(samples)))
    ax2.set_xticklabels(samples, rotation=45, ha='right')
    ax2.grid(False)
    ax2.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    
    # Add value labels on bars
    for i, (bar, val) in enumerate(zip(bars2, alpha_values)):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                f'{val:.2f}', ha='center', va='bottom', fontsize=9)
    
    # Plot 3: R² vs Number of bins
    r_squared = df_results['r_squared']
    n_bins = df_results['n_bins']
    
    scatter = ax3.scatter(n_bins, r_squared, c=colors, s=100, alpha=0.7, edgecolor='black')
    ax3.set_xlabel('Number of Redshift Bins')
    ax3.set_ylabel('R² (Goodness of Fit)')
    ax3.set_title('Fit Quality vs Data Coverage')
    ax3.grid(False)
    
    # Add sample labels
    for i, sample in enumerate(samples):
        ax3.annotate(sample, (n_bins.iloc[i], r_squared.iloc[i]), 
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    # Plot 4: Median mass vs evolution parameter
    median_mass = df_results['median_mass']
    
    scatter = ax4.scatter(median_mass, alpha_values, c=colors, s=100, alpha=0.7, edgecolor='black')
    ax4.errorbar(median_mass, alpha_values, yerr=alpha_errors, fmt='none', color='gray', alpha=0.5)
    ax4.set_xlabel('Median Stellar Mass (log₁₀ M☉)')
    ax4.set_ylabel('α (size evolution parameter)')
    ax4.set_title('Size Evolution vs Stellar Mass')
    ax4.grid(False)
    ax4.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    
    # Add sample labels
    for i, sample in enumerate(samples):
        ax4.annotate(sample, (median_mass.iloc[i], alpha_values.iloc[i]), 
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    plt.tight_layout()
    plt.show()
    
    # Print detailed analysis
    print("\nDetailed Analysis:")
    print("=" * 50)
    for _, row in df_results.iterrows():
        sample = row['sample']
        A, A_err = row['A'], row['A_err']
        alpha, alpha_err = row['alpha'], row['alpha_err']
        r_squared = row['r_squared']
        n_bins = row['n_bins']
        median_mass = row['median_mass']
        
        print(f"\n{sample}:")
        print(f"  Median stellar mass: {median_mass:.2f} log₁₀(M☉)")
        print(f"  Size evolution: log₁₀(R_e/kpc) = {A:.3f} ± {A_err:.3f} - {alpha:.3f} ± {alpha_err:.3f} × log(1+z)")
        print(f"  R² = {r_squared:.3f}, N_bins = {n_bins}")
        
        # Physical interpretation
        if alpha > alpha_err:
            print(f"  → Significant size growth over cosmic time (α > 0)")
        elif alpha < -alpha_err:
            print(f"  → Significant size shrinking over cosmic time (α < 0)")
        else:
            print(f"  → No significant size evolution (α ≈ 0)")
        
        # Expected sizes at different redshifts
        z_values = [0.5, 1.0, 2.0, 3.0]
        print(f"  Expected sizes at different redshifts:")
        for z in z_values:
            log_size = A - alpha * np.log10(1 + z)
            size_kpc = 10**log_size
            print(f"    z={z}: R_e = {size_kpc:.1f} kpc")
